<a href="https://colab.research.google.com/github/PravarGarg/Anemia-Detection-MedGammaGoogle/blob/main/MedGamma_ZeroShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Zero Shot baseline results -
# The MedGamma model cannot predict anemia severity without finetuning - all results all biased towards the majority class 'None'
# Hence, leading to an accuracy of 40%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install transformers accelerate bitsandbytes Pillow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

In [ ]:
import pandas as pd

# Load master.csv from Google Drive
df = pd.read_csv('/content/drive/MyDrive/anemia-detection/master.csv', keep_default_na=False)

# Use only test split for baseline evaluation
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(test_df.shape)
print(test_df[['Number', 'severity']].head(10))

(15, 5)
   Number  severity
0      29      None
1       5    Severe
2      59  Moderate
3      86  Moderate
4      27      None
5      17      Mild
6      79      None
7      30      Mild
8      66      None
9      56  Moderate


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit = True)

model_id = "google/medgemma-4b-it"

processor = AutoProcessor.from_pretrained(model_id, token = HF_TOKEN)

model = AutoModelForImageTextToText.from_pretrained(model_id, quantization_config = bnb_config, device_map = "auto", token = HF_TOKEN)



processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [ ]:
from PIL import Image

first_row = test_df.iloc[0]
number = first_row['Number']
actual_severity = first_row['severity']

image_first_row = Image.open(f"/content/drive/MyDrive/anemia-detection/images/{number}.jpg")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_first_row},
            {"type": "text", "text": "This is a palpebral conjunctiva image. Classify the anemia severity as one of: Severe, Moderate, Mild, or None. Answer with one word only."}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
).to("cuda")

input_len = inputs["input_ids"].shape[1]
generated_tokens = output[0][input_len:]
print("Actual:", actual_severity)
print("Predicted:", processor.decode(generated_tokens, skip_special_tokens=True))

Actual: None
Predicted: None



In [19]:
# cell 8 - to determine the baseline performance

results = []

for idx, rows in test_df.iterrows():
    numbers = rows['Number']
    actual_severity = rows['severity']
    images = Image.open(f"/content/drive/MyDrive/anemia-detection/images/{numbers}.jpg")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": images},
                {"type": "text", "text": "This is a palpebral conjunctiva image. Classify the anemia severity as one of: Severe, Moderate, Mild, or None. Answer with one word only."}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    output = model.generate(**inputs, max_new_tokens=100)

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_len:]
    predicted = processor.decode(generated_tokens, skip_special_tokens=True).strip()

    results.append({'number': numbers, 'actual': actual_severity, 'predicted': predicted})
    print(f"Patient {numbers} | Actual: {actual_severity} | Predicted: {predicted}")

results_df = pd.DataFrame(results)
correct = (results_df['actual'] == results_df['predicted']).sum()
print(f"\nBaseline Accuracy: {correct}/{len(results_df)} = {correct/len(results_df)*100:.1f}%")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Patient 29 | Actual: None | Predicted: None
Patient 5 | Actual: Severe | Predicted: None
Patient 59 | Actual: Moderate | Predicted: None
Patient 86 | Actual: Moderate | Predicted: None
Patient 27 | Actual: None | Predicted: None
Patient 17 | Actual: Mild | Predicted: None
Patient 79 | Actual: None | Predicted: None
Patient 30 | Actual: Mild | Predicted: None
Patient 66 | Actual: None | Predicted: None
Patient 56 | Actual: Moderate | Predicted: None
Patient 70 | Actual: Moderate | Predicted: None
Patient 37 | Actual: None | Predicted: None
Patient 77 | Actual: Moderate | Predicted: None
Patient 53 | Actual: None | Predicted: None
Patient 50 | Actual: Moderate | Predicted: None

Baseline Accuracy: 6/15 = 40.0%
